# Text Summarization Using Supervised Learning (Extractive)

This notebook trains a small **extractive** summarizer (sentence-level Logistic Regression) and lets you try your own text. It's designed to run in any Jupyter environment. If a dependency isn't available, the notebook will attempt to install it or will give clear instructions.

In [ ]:
# --- Dependency check ---
import sys
missing = []
try:
    import sklearn
    import numpy as np
    import pandas as pd
except Exception:
    try:
        import importlib
        for pkg in ('sklearn','numpy','pandas'):
            if importlib.util.find_spec(pkg) is None:
                missing.append(pkg)
    except Exception:
        missing = ['scikit-learn','numpy','pandas']

if missing:
    print('Missing packages detected:', missing)
    print('Attempting to install them. If installation fails, please run the appropriate pip/conda command in your environment.')
    try:
        import subprocess
        to_install = [ 'scikit-learn' if p=='sklearn' else p for p in missing ]
        subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + to_install)
        print('Installation attempt finished. You may need to restart the kernel and re-run the notebook.')
    except Exception:
        print('Automatic installation failed or is not allowed in this environment.')
        print('Please install packages manually:')
        print('  pip install scikit-learn numpy pandas')
        print('Or use conda:')
        print('  conda install scikit-learn numpy pandas')


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd
import re

def simple_sent_tokenize(text):
    parts = re.split(r'(?<=[\.!?])\s+', text.strip())
    return [p.strip() for p in parts if p.strip()]

def simple_word_tokenize(text):
    return re.findall(r"\w+", text)

def normalize_text(s):
    s = s.lower().strip()
    s = re.sub(r'\s+', ' ', s)
    return s


In [ ]:
dataset = [
    {
        'doc': ('Machine learning is the study of computer algorithms that improve automatically through experience. '
                'It is seen as a part of artificial intelligence. Machine learning algorithms build a model based on sample data, '
                'known as training data, in order to make predictions or decisions without being explicitly programmed to do so. '
                'Applications of machine learning include email filtering, speech recognition, and computer vision.'),
        'summary': ('Machine learning is the study of computer algorithms that improve automatically through experience. '
                    'Machine learning algorithms build a model based on sample data, known as training data, in order to make predictions or decisions without being explicitly programmed to do so.')
    },
    {
        'doc': ('Climate change includes both global warming driven by human emissions of greenhouse gases and the resulting large-scale shifts in weather patterns. '
                'Although there have been previous periods of climatic change, since the mid-20th century humans have had an unprecedented impact on Earth\'s climate system and caused change on a global scale. '
                'The effects include rising sea levels, more extreme weather events, and impacts on biodiversity.'),
        'summary': ('Climate change includes both global warming driven by human emissions of greenhouse gases and the resulting large-scale shifts in weather patterns. '
                    'The effects include rising sea levels, more extreme weather events, and impacts on biodiversity.')
    },
    {
        'doc': ('Photosynthesis is a process used by plants and other organisms to convert light energy into chemical energy that can later be released to fuel the organism\'s activities. '
                'This chemical energy is stored in carbohydrate molecules, such as sugars, which are synthesized from carbon dioxide and water. '
                'Oxygen is released as a by-product of photosynthesis.'),
        'summary': ('Photosynthesis is a process used by plants and other organisms to convert light energy into chemical energy that can later be released to fuel the organism\'s activities. '
                    'Oxygen is released as a by-product of photosynthesis.')
    },
    {
        'doc': ('A sorted array is an array where the elements are in order (either ascending or descending). '
                'Searching in a sorted array can be done efficiently using binary search which works in O(log n) time. '
                'Binary search repeatedly divides the search interval in half until the target value is found or the interval is empty.'),
        'summary': ('Searching in a sorted array can be done efficiently using binary search which works in O(log n) time. '
                    'Binary search repeatedly divides the search interval in half until the target value is found or the interval is empty.')
    }
]
train_data, test_data = train_test_split(dataset, test_size=0.25, random_state=42)
print('Train docs:', len(train_data), 'Test docs:', len(test_data))


In [ ]:
def label_sentences(doc, summary):
    doc_sents = simple_sent_tokenize(doc)
    summary_sents = simple_sent_tokenize(summary)
    norm_summary = set(normalize_text(s) for s in summary_sents)
    labels = [1 if normalize_text(s) in norm_summary else 0 for s in doc_sents]
    return doc_sents, labels

sentences, labels, doc_texts, positions, lengths = [], [], [], [], []
for item in train_data:
    sents, labs = label_sentences(item['doc'], item['summary'])
    for i, (s, l) in enumerate(zip(sents, labs)):
        sentences.append(s)
        labels.append(l)
        doc_texts.append(item['doc'])
        positions.append(i / max(1, len(sents)-1))
        lengths.append(len(simple_word_tokenize(s)))

print('Total sentences in training set:', len(sentences))


In [ ]:
tfidf = TfidfVectorizer(ngram_range=(1,2), min_df=1)
all_texts = sentences + doc_texts
tfidf_matrix = tfidf.fit_transform(all_texts)
sent_vecs = tfidf_matrix[:len(sentences)]
doc_vecs = tfidf_matrix[len(sentences):]

sim_scores = [cosine_similarity(sent_vecs[i], doc_vecs[i])[0][0] for i in range(len(sentences))]
X = np.vstack([positions, lengths, sim_scores]).T
y = np.array(labels)

clf = LogisticRegression(max_iter=1000)
clf.fit(X, y)
print('Model trained. Coeff shape:', clf.coef_.shape)


In [ ]:
def summarize_extractively(doc, model, tfidf, max_sentences=2):
    sents = simple_sent_tokenize(doc)
    if len(sents) == 0:
        return ''
    positions = [i / max(1, len(sents)-1) for i in range(len(sents))]
    lengths = [len(simple_word_tokenize(s)) for s in sents]
    sent_tfidf = tfidf.transform(sents)
    doc_tfidf = tfidf.transform([doc]).toarray()
    sim = [cosine_similarity(sent_tfidf[i], doc_tfidf)[0][0] for i in range(sent_tfidf.shape[0])]
    X_new = np.vstack([positions, lengths, sim]).T
    probs = model.predict_proba(X_new)[:,1]
    ranked_idx = np.argsort(-probs + 1e-6 * np.array(positions))
    chosen = sorted(ranked_idx[:max_sentences])
    summary = ' '.join([sents[i] for i in chosen])
    return summary

def rouge_1(reference, generated):
    ref_tokens = simple_word_tokenize(normalize_text(reference))
    gen_tokens = simple_word_tokenize(normalize_text(generated))
    if len(gen_tokens)==0 or len(ref_tokens)==0:
        return {'p':0.0,'r':0.0,'f':0.0}
    ref_counts = {}
    for t in ref_tokens:
        ref_counts[t] = ref_counts.get(t,0)+1
    match = 0
    for t in gen_tokens:
        if ref_counts.get(t,0) > 0:
            match += 1
            ref_counts[t] -= 1
    p = match / len(gen_tokens)
    r = match / len(ref_tokens)
    f = 2*p*r/(p+r) if (p+r)>0 else 0.0
    return {'p':p,'r':r,'f':f}


In [ ]:
results = []
for item in test_data:
    gen_summary = summarize_extractively(item['doc'], clf, tfidf, max_sentences=2)
    rouge = rouge_1(item['summary'], gen_summary)
    results.append({
        'document': item['doc'],
        'reference_summary': item['summary'],
        'generated_summary': gen_summary,
        'rouge1_precision': round(rouge['p'],3),
        'rouge1_recall': round(rouge['r'],3),
        'rouge1_f1': round(rouge['f'],3)
    })

import pandas as pd
df = pd.DataFrame(results)
df


In [ ]:
print('=== Try your own paragraph ===')
try:
    user_doc = input('Paste a paragraph here and press Enter:\n')
except Exception:
    user_doc = ''

if user_doc.strip():
    generated = summarize_extractively(user_doc, clf, tfidf, max_sentences=2)
    print('\nGenerated summary:\n')
    print(generated)
else:
    print('\nNo input detected. To test, set the `user_doc` variable to a string and re-run this cell.')


## Abstract

This notebook implements a simple supervised extractive summarization pipeline. Sentences are labeled as salient or not, features (position, length, and TF–IDF similarity to the document) are computed, and a Logistic Regression classifier is trained to predict sentence salience. Summaries are produced by selecting top-scoring sentences. The notebook is intentionally lightweight so it runs in constrained environments and provides a clear baseline for further improvements such as richer features or transformer-based abstractive models.